# 03_lcel_chains

对应 `index.md` 第 3 阶段：LCEL `|` 组链、Runnable。

笔记本在 `codes/langchain/03_lcel_chains/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "03_lcel_chains" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "03_lcel_chains"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/03_lcel_chains')

## 1. 用 `|` 串三条

已有 `prompt`、`model`、`parser`。用 `|` 组成链：`prompt | model | parser`。

`invoke({"topic": "咖啡"})`，打印结果（应是字符串）。


In [3]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([("human", "用一个词形容 {topic}")])
model = FakeListChatModel(responses=["香"])
parser = StrOutputParser()

# 作答

chain = prompt | model | parser

print(chain.invoke({"topic": "咖啡"}))
#评阅
# 对。prompt | model | parser，结果是字符串「香」。

#参考答案
# chain = prompt | model | parser
# print(chain.invoke({"topic": "咖啡"}))

香


## 2. RunnableLambda

用 `RunnableLambda` 包一个函数：把输入字符串转成大写并加上后缀 `!`。

对输入字符串 hello 做 `invoke`，打印结果。


In [4]:
from langchain_core.runnables import RunnableLambda

# 作答

def operate_str(input: str) -> str:
    return input.upper() + "!"

runnable = RunnableLambda(operate_str)

print(runnable.invoke("hello"))
#评阅
# 对。大写加后缀，得到 HELLO!。

#参考答案
# print(RunnableLambda(lambda s: s.upper() + "!").invoke("hello"))

HELLO!


## 3. Passthrough.assign 补字段

用 `RunnablePassthrough.assign`：保留原字典，并新增键 `len`，值为 `len(x["text"])`。

对 `{"text": "abcd"}` 做 `invoke`，打印结果字典。


In [5]:
from langchain_core.runnables import RunnablePassthrough

# 作答

runnable = RunnablePassthrough().assign(len = lambda x: len(x["text"]))

print(runnable.invoke({"text": "abcd"}))
#评阅
# 对。保留 text，并加上 len=4。

#参考答案
# r = RunnablePassthrough.assign(len=lambda x: len(x["text"]))
# print(r.invoke({"text": "abcd"}))

{'text': 'abcd', 'len': 4}


## 4. batch 一批输入

已有链 `chain`。对列表 `[{"n": 1}, {"n": 2}, {"n": 3}]` 调用 `batch`，打印返回的列表。


In [8]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([("human", "数字 {n}")])
model = FakeListChatModel(responses=["一", "二", "三"])
chain = prompt | model | StrOutputParser()

# 作答

print(chain.batch([{"n": 1}, {"n": 2}, {"n": 3}]))
#评阅
# 对。batch 得到 ['一', '二', '三']。

#参考答案
# print(chain.batch([{"n": 1}, {"n": 2}, {"n": 3}]))

['一', '二', '三']


## 5. 两段链再拼接

已有 `prep`（把 `{"q": ...}` 变成 `{"topic": ...}`）和 `tail`（prompt|model|parser）。

用 `|` 把 `prep | tail` 接成完整链，对 `{"q": "雨"}` 做 `invoke`，打印结果。


In [9]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

prep = RunnableLambda(lambda d: {"topic": d["q"]})
prompt = ChatPromptTemplate.from_messages([("human", "一个字写 {topic}")])
model = FakeListChatModel(responses=["湿"])
tail = prompt | model | StrOutputParser()

# 作答

chain = prep | tail

print(chain.invoke({"q": "雨"}))
#评阅
# 对。prep | tail，结果「湿」。

#参考答案
# print((prep | tail).invoke({"q": "雨"}))

湿
